# جعبه‌ابزار پایتون و NumPy برای تمام ترم

**مخاطب:** دانشجویی که متغیر، شرط، حلقه، لیست و تعریف ساده تابع را می‌شناسد؛ اما هنوز در خواندن کدهای پردازش داده روان نیست.

### خروجی قابل انتظار
- بتوانید `obj[i]`، `obj(x)`، `*args` و `**kwargs` را بخوانید.
- قبل از اجرای هر عملیات، `shape` خروجی را پیش‌بینی کنید.
- داده و برچسب را با یک اندیس مشترک انتخاب، فیلتر و shuffle کنید.
- تفاوت copy و view و خطر تغییر ناخواسته داده را بشناسید.
- یک جریان کوچک از تمیزکاری تا استانداردسازی، batch و ذخیره‌سازی بسازید.

### شیوه استفاده
در Jupyter یا Colab باز کنید. از منوی Kernel / Runtime، گزینه Restart و سپس Run All را اجرا کنید. فقط NumPy لازم است؛ سلول نصب زیر عمداً کامنت است. سلول‌های خطای عمدی، خطا را مدیریت می‌کنند تا اجرای کامل قطع نشود. فایل‌های خروجی در `bootcamp_artifacts` ساخته می‌شوند.

In [ ]:
# فقط در صورت نبود NumPy، در یک سلول جدا اجرا کنید:
# %pip install numpy
import sys
from pathlib import Path
import numpy as np
np.set_printoptions(precision=3, suppress=True)
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__)

def expected_error(error_type, operation):
    """خطای آموزشی را نشان بده؛ اگر خطای مورد انتظار رخ نداد، تست شکست بخورد."""
    try:
        operation()
    except error_type as error:
        print(type(error).__name__ + ':', str(error).splitlines()[0])
    else:
        raise AssertionError('Expected error did not occur')

Python: 3.13.18 | NumPy: 2.1.3


## ۱. پایتون میانی — اجرای زنده، دقیقه ۰ تا ۲۰

**پرسش شروع:** فرق `=` و `==` چیست؟ یک تابع چگونه مقدار برمی‌گرداند؟ اگر این دو سؤال مبهم‌اند، پیوست صفر را ابتدا بخوانید.

در همه مثال‌ها، هر سطر یک نمونه است. نام‌های `X` و `y` به‌ترتیب ویژگی‌ها و برچسب‌ها را نشان می‌دهند. همین قرارداد را تا آخر ترم ثابت نگه دارید.

### ۱.۱ — پیمایش هم‌زمان و باز کردن بسته‌ها

`enumerate` شمارنده می‌دهد؛ `zip` اعضای هم‌موقعیت را کنار هم می‌گذارد. در `for i, (row, label)` ساختار داخل پرانتز متناظر با ساختار خروجی است. شمارنده از صفر شروع می‌شود مگر `start` بدهیم. ستاره در unpacking، باقی اعضا را در یک **لیست** جمع می‌کند؛ در هر سطح فقط یک هدف ستاره‌دار مجاز است.

In [ ]:
names = ['Sara', 'Reza', 'Nima']
scores = [17, 12, 19]
for i, (name, score) in enumerate(zip(names, scores), start=1):
    print(f'{i}: {name} -> {score}')
    print(f"{i}")
record = ('station-A', 12.0, 14.0, 18.0, 'valid')
station, *measurements, status = record
print(station, measurements, status)
first, second = 3, 8
first, second = second, first
print('swapped:', first, second)

1: Sara -> 17
2: Reza -> 12
3: Nima -> 19
station-A [12.0, 14.0, 18.0] valid
swapped: 8 3


### ۱.۲ — تله zip و تفاوت فیلتر با تبدیل

`zip` معمولی در کوتاه‌ترین ورودی متوقف می‌شود؛ این رفتار می‌تواند آخر داده را بی‌صدا حذف کند. برای جفت داده/برچسب طول را کنترل کنید. Python 3.10 به بعد `zip(..., strict=True)` هم دارد. comprehension با `if` انتهایی نمونه‌ها را حذف می‌کند؛ عبارت شرطی قبل از `for` طول را حفظ می‌کند.

In [ ]:
scores = [17, 12, 19, 20]
names = ["a", "b", "c"]

list(zip(names, scores))

[('a', 17), ('b', 12), ('c', 19)]

In [ ]:
print('silent truncation:', list(zip([1, 2, 3], ['a', 'b'])))
assert len(names) == len(scores)
raw = [-3, 0, 2, 5]
filtered = [x**2 for x in raw if x > 0]
replaced = [x if x > 0 else 0 for x in raw]
column_index = {name: i for i, name in enumerate(['temp', 'pressure', 'target'])}
print('filter:', filtered, '| replace:', replaced)
print('mapping:', column_index)
print('items:', list(column_index.items()))

### ۱.۳ — امضای تابع، return و باز کردن آرگومان‌ها

`*args` آرگومان‌های موقعیتی اضافه را به tuple و `**kwargs` آرگومان‌های نام‌دار اضافه را به dict تبدیل می‌کند. در محل فراخوانی برعکس عمل می‌کنند: `*values` یک دنباله و `**config` یک نگاشت را باز می‌کند. نام‌های args و kwargs قراردادی‌اند؛ معنای اصلی در ستاره‌هاست. پارامتر بعد از `*` می‌تواند فقط با نام ارسال شود.

In [ ]:
def f1(x):
    print(x+10)

print(f1(5)*2)

15


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

In [ ]:
def summation(*x):
    return sum(x)
summation(1, 2, 3, 4, 5)

15

In [ ]:

x = 10
x

10

In [ ]:
x+13

23

In [ ]:
def summarize(values, *, digits=2):
    return round(sum(values) / len(values), digits)

def report(title, *metrics, **options):
    return {'title': title, 'metrics': metrics, 'options': options}

config = {'digits': 1}
print(summarize([10, 12, 15], **config))
metrics = ['count', 'mean']
print(report('run-01', *metrics, batch_size=4, shuffle=True), "here")
def pair(a, b):
    return a + b, a - b
total, difference = pair(*[7, 2])
print(total, difference)

12.3
{'title': 'run-01', 'metrics': ('count', 'mean'), 'options': {'batch_size': 4, 'shuffle': True}} here
9 5


In [ ]:
def f1(x, *y, digit = 5):
    return x + sum(y) + digit


y = [1, 2, 3]
f1(2, 3, 2, 1, 10, digit = 2)

20

### ۱.۴ — lambda برای یک کار کوچک

`lambda` یک عبارت و یک مقدار خروجی دارد و برای کلید مرتب‌سازی کوتاه مناسب است. برای چند مرحله، نام معنادار و توضیح، `def` خواناتر است. `sorted` یک لیست جدید می‌دهد؛ ترتیب ورودی را تغییر نمی‌دهد.

In [ ]:
list1 = [1, 10, 5]
list1 = sorted(list1, key= lambda item: -item)
list1

[10, 5, 1]

In [ ]:
records = [('Sara', 17), ('Reza', 12), ('Nima', 19)]
ranked = sorted(records, key=lambda item: item[1], reverse=True)
print(ranked)
def score_key(item):
    return item[1]
assert ranked == sorted(records, key=score_key, reverse=True)

### ۱.۵ — کلاس به‌عنوان داده + رفتار

کلاس یک قالب است؛ هر نمونه وضعیت خودش را دارد. `self` همان نمونه‌ای است که متد روی آن اجرا می‌شود. `__init__` هنگام ساخت، `__len__` با `len(obj)`، `__getitem__` با `obj[i]` و `__call__` با `obj(x)` درگیر می‌شوند. این‌ها قراردادهای زبان‌اند؛ جادوی مخصوص یادگیری ماشین نیستند. برای روشن ماندن مسئولیت‌ها، dataset و تبدیل را دو کلاس جدا می‌سازیم.

In [ ]:
class TinyDataset:
    def __init__(self, rows, labels):
        if len(rows) != len(labels):
            raise ValueError('rows and labels must have equal lengths')
        self.rows = rows
        self.labels = labels

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index], self.labels[index]

class AffineTransform:
    def __init__(self, scale=1.0, offset=0.0):
        self.scale = scale
        self.offset = offset

    def __call__(self, x):
        return self.scale * x + self.offset

dataset = TinyDataset([[1, 2], [3, 4], [5, 6]], [0, 1, 0])
transform = AffineTransform(scale=2, offset=1)
print('length:', len(dataset), '| item:', dataset[1])
print('call:', transform(3))
print('another instance:', AffineTransform(scale=10)(3))

In [ ]:
class c1:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return 1
    def __getitem__(self, index):
        return index+4
    def __call__(self, index):
        return index+7
    def __matmul__(self, other):
        return other
    def summation(self):
        return self.x + self.y


o1 = c1(1, 4)
o1.summation()
# len(o1), o1[10], o1(10), o1 @ 6

len(o1)
o1[2]
o1(6)

o1 @ 12

12

In [ ]:
x = [1, 2, 3]
len(x)

x[1]

2

### تمرین 1 — فیلتر و شیء قابل فراخوانی — ۳ دقیقه

با `zip` و dict comprehension نام کسانی را نگه دارید که نمره‌شان حداقل ۱۵ است. سپس کلاس `ThresholdFilter` بسازید که با فراخوانی روی یک لیست، فقط مقادیر بزرگ‌تر یا مساوی آستانه را برگرداند. خروجی مورد انتظار دیکشنری: `{'Sara': 17, 'Nima': 19}`؛ فیلتر آستانه ۱۰: `[12, 10]`.

In [ ]:
ex1_names = ['Sara', 'Reza', 'Nima']
ex1_scores = [17, 12, 19]
# TODO: passed = ...
# TODO: class ThresholdFilter ...

## ۲. آرایه و مدیریت شکل — اجرای زنده، دقیقه ۲۰ تا ۳۵

**قرارداد ثابت:** جدول `(N, F)` یعنی N نمونه و F ویژگی؛ دنباله `(N, T, F)` یعنی نمونه، زمان و ویژگی. نام محور را کنار شکل بنویسید؛ صرف حفظ کردن عددها کافی نیست.

### ۲.۱ — لیست با ndarray یکی نیست

لیست یک ظرف عمومی است؛ ndarray عددی معمولاً نوع داده مشترک دارد. `shape` اندازه هر محور، `ndim` تعداد محورها، `size` تعداد کل درایه‌ها و `dtype` نوع هر درایه است. نوع داده روی دقت و حافظه اثر دارد. آرایه عدد صحیح برای نگه‌داشتن اعشار مناسب نیست.

In [ ]:
import numpy as np
v = np.array([1, 2, 3]).astype("float32")
u = [1, 2, 3]

print(v*3)



[3. 6. 9.]
[3. 6. 9.]


In [ ]:
print('list repetition:', [1, 2, 3] * 2)
print('array multiplication:', np.array([1, 2, 3]) * 2)
A = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
print(A)

print('shape / ndim / size / dtype / bytes:', A.shape, A.ndim, A.size, A.dtype, A.nbytes)
integer_array = np.array([1, 2, 3])
integer_array[0] = 1.99999
print('integer assignment:', integer_array)
print('cast:', integer_array.astype(np.float64))

list repetition: [1, 2, 3, 1, 2, 3]
array multiplication: [2 4 6]
[[1. 2. 3.]
 [4. 5. 6.]]
shape / ndim / size / dtype / bytes: (2, 3) 2 6 float32 24
integer assignment: [1 2 3]
cast: [1. 2. 3.]


### ۲.۲ — ساخت آرایه و تصادفیِ قابل بازتولید

`arange` معمولاً انتهای باز و `linspace` به‌طور پیش‌فرض انتهای بسته دارد. برای کد جدید یک generator محلی با `default_rng` می‌سازیم. `np.random.seed` را هم برای خواندن کدهای قدیمی می‌بینیم؛ این دو مسیر وضعیت تصادفی یکسانی ندارند. seed برابر به معنی برابری خروجی همه APIها نیست.

In [ ]:
print(np.zeros((2, 3), dtype=np.float32))
print(np.ones((1, 3)), np.full((2, 2), 7), np.eye(3), sep='\n')
print('arange:', np.arange(0, 10, 2))
print('linspace:', np.linspace(0, 1, 5))
rng = np.random.default_rng(42)
print('normal:', rng.normal(0, 1, size=(2, 3)))
print('uniform:', rng.uniform(-1, 1, size=(2, 3)))
print('integers:', rng.integers(0, 4, size=6))
np.random.seed(42)  # API قدیمی؛ روی rng بالا اثر ندارد
print('legacy:', np.random.randn(2))
np.testing.assert_array_equal(np.random.default_rng(9).integers(0, 100, 5),
                              np.random.default_rng(9).integers(0, 100, 5))

[[0. 0. 0.]
 [0. 0. 0.]]
[[1. 1. 1.]]
[[7 7]
 [7 7]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
normal: [[ 0.305 -1.04   0.75 ]
 [ 0.941 -1.951 -1.302]]
uniform: [[ 0.522  0.572 -0.744]
 [-0.099 -0.258  0.854]]
integers: [3 2 1 3 2 1]
legacy: [ 0.497 -0.138]


### ۲.۳ — reshape، بردار، سطر و ستون

در reshape تعداد کل عناصر ثابت می‌ماند و فقط یک بعد می‌تواند `-1` باشد. `(4,)` نه ماتریس سطری است و نه ستونی؛ `.T` روی آن شکل را عوض نمی‌کند. reshape ترتیب محورهای معنایی را جابه‌جا نمی‌کند؛ برای آن transpose لازم داریم.

In [ ]:
sequence = np.arange(24)
print(sequence.reshape(4, -1))
print('3D:', sequence.reshape(2, 3, -1).shape)
v = np.arange(4)
print('vector / transpose:', v.shape, v.T.shape)
v2 = v.reshape(1, -1)
print('vector / transpose:', v2.shape, v2.T.shape)
print('row / column:', v[None, :].shape, v[:, None].shape)
print('expand:', np.expand_dims(v, axis=0).shape)
expected_error(ValueError, lambda: sequence.reshape(5, -1))
sensor_sequences = np.arange(5 * 4 * 3).reshape(5, 4, 3)
print('flatten each sample:', sensor_sequences.reshape(len(sensor_sequences), -1).shape)

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]]
3D: (2, 3, 4)
vector / transpose: (4,) (4,)
vector / transpose: (1, 4) (4, 1)
row / column: (1, 4) (4, 1)
expand: (1, 4)
ValueError: cannot reshape array of size 24 into shape (5,newaxis)
flatten each sample: (5, 12)


In [ ]:
x = np.array([1, 2])
y = np.array([[3], [4], [5]])

x = np.reshape(x, (1, -1))

print(x)
print(y)

x.shape, y.shape


x+y

[[1 2]]
[[3]
 [4]
 [5]]


array([[4, 5],
       [5, 6],
       [6, 7]])

### ۲.۴ — squeeze، flatten و اتصال

`squeeze` فقط محورهای طول یک را حذف می‌کند؛ حذف همه آن‌ها می‌تواند محور batch تک‌نمونه را هم بردارد. با `axis` هدف را صریح کنید. `flatten` کپی مستقل می‌سازد؛ `ravel` و `reshape` بسته به چیدمان حافظه ممکن است view یا copy باشند. `concatenate` محور موجود را بلندتر و `stack` محور جدید ایجاد می‌کند.

In [ ]:
x = np.arange(5)

print(x.shape)

x = x.reshape(1, -1)
print(x.shape)

np.squeeze(x).shape

(5,)
(1, 5)


(5,)

In [ ]:

z = sequence.reshape(4, -1)

z.flatten()





array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23])

In [ ]:

np.ones((4, 5))

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [ ]:
single_batch = np.zeros((1, 4, 1))
print('all singleton axes:', single_batch.squeeze().shape)
print('only last axis:', np.squeeze(single_batch, axis=-1).shape)
left = np.ones((2, 3)); right = np.zeros((2, 3))
print()
print("left:\n", left)
print("right:\n", right)
print()
print('concat rows:', np.concatenate([left, right], axis=0).shape)
print('concat columns:', np.concatenate([left, right], axis=1).shape)
print('vstack / hstack:', np.vstack([left, right]).shape, np.hstack([left, right]).shape)
print('stack:', np.stack([left, right], axis=0).shape)
pieces = np.split(np.arange(12), 3)
print('equal split:', pieces, type(pieces))
print('unequal split:', [x.shape for x in np.array_split(np.arange(10), 3)])

all singleton axes: (4,)
only last axis: (1, 4)

left:
 [[1. 1. 1.]
 [1. 1. 1.]]
right:
 [[0. 0. 0.]
 [0. 0. 0.]]

concat rows: (4, 3)
concat columns: (2, 6)
vstack / hstack: (4, 3) (2, 6)
stack: (2, 2, 3)
equal split: [array([0, 1, 2, 3]), array([4, 5, 6, 7]), array([ 8,  9, 10, 11])] <class 'list'>
unequal split: [(4,), (3,), (3,)]


In [ ]:
tmp_arr = np.zeros((1, 4))
tmp_arr.shape, tmp_arr

((1, 4), array([[0., 0., 0., 0.]]))

### ۲.۵ — alias، view و copy

`b = a` همان شیء را نام‌گذاری می‌کند. slicing پایه معمولاً view از همان حافظه می‌دهد؛ تغییر آن به اصل منتقل می‌شود. `.copy()` داده را مستقل می‌کند. با `np.shares_memory` آزمایش کنید؛ از مشاهده یک مثال، حکم کلی درباره همه reshapeها ندهید.

In [ ]:
original = np.arange(12).reshape(3, 4)
alias = original
window = original[:, :2]
independent = window.copy()
flat_copy = original.flatten()
print('same object:', alias is original)
print('shared:', np.shares_memory(original, window), np.shares_memory(original, independent))
window[0, 0] = 999
independent[0, 0] = -5
print('original:', original, sep='\n')
assert original[0, 0] == 999 and flat_copy[0] == 0



same object: True
shared: True False
original:
[[999   1   2   3]
 [  4   5   6   7]
 [  8   9  10  11]]


In [ ]:
x = 12
assert x==12

### تمرین 2 — دنباله‌ها به جدول — ۳ دقیقه

آرایه `(6,4,3)` را به `(6,12)` تبدیل کنید؛ یک ستون یک اضافه کنید تا `(6,13)` شود. سپس از دو ستون اول کپی مستقلی بسازید و نشان دهید تغییر کپی اصل را تغییر نمی‌دهد.

In [ ]:
ex2_sequences = np.arange(72).reshape(6, 4, 3)
# TODO: flat = ...
# TODO: augmented = ...
# TODO: independent_columns = ...

## ۳. انتخاب هوشمند داده — اجرای زنده، دقیقه ۳۵ تا ۵۰

در جدول زیر سه ستون ویژگی حسگر و ستون آخر برچسب است. برای آموزش جدول یکپارچه ساخته‌ایم؛ در برنامه واقعی بهتر است ویژگی اعشاری و برچسب صحیح جدا نگه داشته شوند.

In [ ]:
table = np.array([
    [10,  2.5, -1.0, 0],
    [15, -3.2,  4.1, 1],
    [ 8,  0.5, -2.2, 0],
    [22,  5.0,  1.8, 1],
    [18,  1.5,  3.0, 1],
    [11, -0.5,  2.0, 0],
], dtype=float)
X = table[:, :-1].copy()
y = table[:, -1].astype(np.int64)
sample_ids = np.arange(len(X))
print('X:', X.shape, '| y:', y.shape, y.dtype)

X: (6, 3) | y: (6,) int64


In [ ]:
print(X)

[[10.   2.5]
 [15.  -3.2]
 [ 8.   0.5]
 [22.   5. ]
 [18.   1.5]
 [11.  -0.5]]


In [ ]:
x = [1, 2, 3, 4, 5]
x[-1], x[:-2]

(5, [1, 2, 3])

### ۳.۱ — slicing و باقی ماندن بعد

فرم عمومی `start:stop:step` است؛ stop وارد خروجی نمی‌شود. اندیس صحیح معمولاً یک محور را حذف می‌کند؛ slice طول یک آن را نگه می‌دارد. محور منفی از آخر شمرده می‌شود. برعکس کردن سطرها با برعکس کردن ستون‌ها فرق دارد.

In [ ]:
print(X, "\n")
print('first two rows:', X[:2])
X[start:stop:step]
print('every other row:', X[::2])
print('reverse rows:', X[::-1])
print('last column as vector:', X[:, -1].shape)
print('last column as matrix:', X[:, -1:].shape)
print('single row / batch of one:', X[0].shape, X[0:1].shape)
print('drop last feature:', X[:, :-1].shape)

[[10.   2.5 -1. ]
 [15.  -3.2  4.1]
 [ 8.   0.5 -2.2]
 [22.   5.   1.8]
 [18.   1.5  3. ]
 [11.  -0.5  2. ]] 

first two rows: [[10.   2.5 -1. ]
 [15.  -3.2  4.1]]
every other row: [[10.   2.5 -1. ]
 [ 8.   0.5 -2.2]
 [18.   1.5  3. ]]
reverse rows: [[11.  -0.5  2. ]
 [18.   1.5  3. ]
 [22.   5.   1.8]
 [ 8.   0.5 -2.2]
 [15.  -3.2  4.1]
 [10.   2.5 -1. ]]
last column as vector: (6,)
last column as matrix: (6, 1)
single row / batch of one: (3,) (1, 3)
drop last feature: (6, 2)


### ۳.۲ — ماسک سطری و ماسک درایه‌ای

ماسک سطری شکل `(N,)` دارد و همه ویژگی‌های سطرهای منتخب را نگه می‌دارد. ماسک هم‌شکل X، درایه‌ها را انتخاب و معمولاً یک خروجی یک‌بعدی می‌دهد. برای ترکیب شرط آرایه از `&` و `|` با پرانتز استفاده کنید؛ `and` و `or` برای این کار نیستند. `.any()` یعنی دست‌کم یک True و `.all()` یعنی همه True.

In [ ]:
a = np.arange(5)
print(a)
a>3



[0 1 2 3 4]


array([False, False, False, False,  True])

In [ ]:
row_mask = (y == 1) & (X[:, 0] >= 15)
selected_X, selected_y = X[row_mask], y[row_mask]
print('row mask:', row_mask, '| selected:', selected_X.shape)
print('element selection:', X[X < 0], X[X < 0].shape)
print('any / all:', (X < 0).any(), (X >= 0).all())
repaired = np.where(X < 0, 0.0, X)
row_indices = np.where(row_mask)[0]
print('where replacement:', repaired)
print('where indices:', row_indices)
assert len(selected_X) == len(selected_y)

row mask: [False  True False  True  True False] | selected: (3, 3)
element selection: [-1.  -3.2 -2.2 -0.5] (4,)
any / all: True False
where replacement: [[10.   2.5  0. ]
 [15.   0.   4.1]
 [ 8.   0.5  0. ]
 [22.   5.   1.8]
 [18.   1.5  3. ]
 [11.   0.   2. ]]
where indices: [1 3 4]


### ۳.۳ — fancy indexing و shuffle امن

آرایه یا لیست اندیس‌ها ترتیب و تکرار دلخواه می‌دهد. advanced indexing هنگام خواندن copy می‌دهد. برای shuffle، فقط یک permutation بسازید و روی X و y و شناسه‌ها اعمال کنید؛ seed ثابت به‌تنهایی جای این قرارداد را نمی‌گیرد.

In [ ]:
print("X:", X)
print("type(X):", type(X), "\n")

indices = np.array([4, 0, 4])
picked = X[indices]
print('repeated selection:', picked)
print('shares memory:', np.shares_memory(picked, X), "\n")

shuffle_rng = np.random.default_rng(7)
order = shuffle_rng.permutation(len(X))
X_shuffled, y_shuffled, ids_shuffled = X[order], y[order], sample_ids[order]

print('order:', order)

np.testing.assert_array_equal(X_shuffled, X[ids_shuffled])
np.testing.assert_array_equal(y_shuffled, y[ids_shuffled])
print('alignment verified')

X: [[10.   2.5 -1. ]
 [15.  -3.2  4.1]
 [ 8.   0.5 -2.2]
 [22.   5.   1.8]
 [18.   1.5  3. ]
 [11.  -0.5  2. ]]
type(X): <class 'numpy.ndarray'> 

repeated selection: [[18.   1.5  3. ]
 [10.   2.5 -1. ]
 [18.   1.5  3. ]]
shares memory: False 

order: [5 2 0 4 1 3]
alignment verified


In [ ]:
Z = np.array([1, 2, 3, 4, 5])

rng = np.random.default_rng(3)
order = rng.permutation(len(Z))
order

Z2 = Z[order]

print(Z)
np.argmax(Z)

[1 2 3 4 5]


np.int64(4)

### ۳.۴ — مقدار بیشینه در برابر محل آن

`max` مقدار می‌دهد؛ `argmax` اندیس روی محور انتخاب‌شده. در تساوی بیشینه، argmax اولین محل را برمی‌گرداند. `argsort` اندیس ترتیب صعودی را می‌دهد. دو اندیس آخر، top-2 را به ترتیب صعودی در همان دو عضو می‌دهند؛ برای ترتیب نزولی برعکس کنید.

In [ ]:
scores_matrix = np.array([[12, 19, 14], [18, 15, 16], [10, 11, 19], [20, 20, 17]])
print('max per row:', scores_matrix.max(axis=1))
best = scores_matrix.argmax(axis=1)
print('best column:', best)
print('values at best columns:', scores_matrix[np.arange(len(scores_matrix)), best])
top2_indices = np.argsort(scores_matrix, axis=1)[:, -2:][:, ::-1]
top2_values = np.take_along_axis(scores_matrix, top2_indices, axis=1)
print('top2 indices:', top2_indices, '| values:', top2_values)

### تمرین 3 — انتخاب برنده و صحت تطابق — ۳ دقیقه

جدول امتیاز زیر چهار گزینه برای هر نمونه دارد. گزینه برنده هر سطر، نسبت تطابق با پاسخ و شناسه سطرهای اشتباه را بدون حلقه بیابید. خروجی‌ها: `[1,0,2,1,3]`، `0.8` و `[2]`. این یک تمرین انتخاب و مقایسه است و به مدل خاصی وابسته نیست.

ابتدا خودتان تلاش کنید؛ پاسخ در سلول بعد از محل کار شماست.

In [ ]:
ex3_scores = np.array([[.1,.7,.1,.1], [.8,.1,.05,.05], [.2,.2,.5,.1],
                       [.3,.4,.2,.1], [0,.1,.1,.8]])
ex3_answers = np.array([1, 0, 3, 1, 3])
# TODO: winners, accuracy, wrong_ids

## ۴. ضرب، محورها و Broadcasting — اجرای زنده، دقیقه ۵۰ تا ۶۵

**قانون توضیح روی تخته:** axis محوری است که روی آن تجمیع می‌کنیم؛ بدون keepdims آن محور از شکل خروجی حذف می‌شود. به‌جای عبارت مبهم «عمودی/افقی»، نام محور حذف‌شونده را بگویید. این زبان برای ۳ و ۴ بعد هم کار می‌کند.

### ۴.۱ — سه ضرب، سه قرارداد

`*` ضرب درایه‌ای با broadcasting است؛ `@` و `np.matmul` ضرب ماتریسی‌اند و محورهای ابتدایی را برای batch در نظر می‌گیرند. `np.dot` برای دو ماتریس همان نتیجه را می‌دهد اما در ابعاد بالاتر قرارداد متفاوت دارد. matmul یک ضرب تانسوری عمومی با محورهای انقباض دلخواه نیست.

In [ ]:
X = np.array([1, 2]).reshape(1, -1)

Y = np.array([[4], [5], [6]])

print("X.shape, Y.shape -->", X.shape, Y.shape)

X+Y

X.shape, Y.shape --> (1, 2) (3, 1)


array([[5, 6],
       [6, 7],
       [7, 8]])

In [ ]:
M = np.array([[1, 2], [3, 4]])
N = np.array([[10, 20], [30, 40]])

print("M:\n", M)
print("N:\n", N)
print()

print('elementwise:', M * N, sep='\n')
print('matrix:', M @ N, "", sep='\n')
print('matrix:', np.matmul(M, N), "", sep='\n')
np.testing.assert_array_equal(M @ N, np.dot(M, N))
np.testing.assert_array_equal(M @ N, np.matmul(M, N))

batch_A = np.ones((5, 2, 3))
batch_B = np.ones((5, 3, 4))
print('batched @:', (batch_A @ batch_B).shape)
print('high-dimensional dot:', np.dot(batch_A, batch_B).shape)
u = np.array([1, 2, 3])
print('vector inner product:', u @ u, '| outer:', (u[:, None] @ u[None, :]).shape)

M:
 [[1 2]
 [3 4]]
N:
 [[10 20]
 [30 40]]

elementwise:
[[ 10  40]
 [ 90 160]]
matrix:
[[ 70 100]
 [150 220]]

batched @: (5, 2, 4)
high-dimensional dot: (5, 2, 5, 4)
vector inner product: 14 | outer: (3, 3)


### ۴.۲ — transpose جابه‌جایی محور است

روی آرایه چندبعدی `.T` ترتیب تمام محورها را معکوس می‌کند. در داده حسگر `(N,T,F)` اگر `(N,F,T)` بخواهیم باید محور batch را سر جایش نگه داریم. `reshape` جایگزین transpose نیست؛ شکل یکسان می‌تواند معنای داده متفاوت داشته باشد.

In [ ]:
signals = np.arange(2 * 4 * 3).reshape(2, 4, 3)
swapped = np.transpose(signals, (0, 2, 1))
print('N,T,F:', signals.shape, '| N,F,T:', swapped.shape, '| .T:', signals.T.shape)
assert swapped[1, 2, 3] == signals[1, 3, 2]
print('same shape, different meaning:', np.array_equal(swapped, signals.reshape(2, 3, 4)))
print('explicit (2,0,1):', np.transpose(signals, (2, 0, 1)).shape)

### ۴.۳ — axis و keepdims با مثال قابل بررسی

برای جدول `(N,F)`، `axis=0` محور نمونه را حذف می‌کند و به ازای هر ویژگی نتیجه می‌دهد؛ `axis=1` محور ویژگی را حذف می‌کند و به ازای هر نمونه نتیجه می‌دهد. `keepdims=True` محور کاهش‌یافته را با اندازه یک نگه می‌دارد تا محل آن روشن بماند.

In [ ]:
D = np.array([[10.,20.,30.], [40.,50.,60.], [70.,80.,90.], [100.,110.,120.]])
print('sum features:', D.sum(axis=0), D.sum(axis=0).shape)
print('mean each sample:', D.mean(axis=1), D.mean(axis=1).shape)
print('std features:', D.std(axis=0, keepdims=True))
expected_error(ValueError, lambda: D - D.mean(axis=1))
row_centered = D - D.mean(axis=1, keepdims=True)
print('kept shape:', D.mean(axis=1, keepdims=True).shape)
print(row_centered)
np.testing.assert_allclose(row_centered.mean(axis=1), 0, atol=1e-12)

### ۴.۴ — قانون Broadcasting از راست

شکل‌ها را از سمت راست هم‌تراز کنید؛ دو بعد باید برابر باشند یا یکی ۱ باشد. محور غایب مانند ۱ است. شکل `(N,)` برای کم‌کردن مقدار مخصوص هر سطر معمولاً کافی نیست؛ `(N,1)` لازم داریم. broadcasting معمولاً ورودی را واقعاً تکرار نمی‌کند، اما خروجی و نتایج میانی می‌توانند بزرگ باشند.

In [ ]:
print('feature offsets:', (D + np.array([1, 2, 3])).shape)
print('sample offsets:', (D + np.arange(4)[:, None]).shape)
print('outer addition:', np.arange(3)[:, None] + np.arange(4)[None, :])
for shape_a, shape_b in [((4,3), (3,)), ((4,3), (4,1)), ((4,3), (4,))]:
    try:
        print(shape_a, shape_b, '->', np.broadcast_shapes(shape_a, shape_b))
    except ValueError:
        print(shape_a, shape_b, '-> incompatible')
# برداری‌سازی: همان نتیجه حلقه، با بیان مستقیم عملیات آرایه‌ای
loop_result = np.array([[value + 2 for value in row] for row in D])
np.testing.assert_array_equal(loop_result, D + 2)

### تمرین 4 — استانداردسازی و قرارداد ابعاد — ۳ دقیقه

برای هر ستون، میانگین و انحراف معیار را با حفظ بعد حساب کنید. ستون ثابت را با مقیاس ۱ مدیریت کنید تا خروجی آن صفر شود. سپس ماتریس استانداردشده را در ماتریس تبدیل `(3,2)` ضرب و offset `(1,2)` اضافه کنید. خروجی `(4,2)` است؛ این صرفاً تبدیل جدول است.

ابتدا خودتان تلاش کنید؛ پاسخ در سلول بعد از محل کار شماست.

In [ ]:
ex4_data = np.array([[1., 10., 7.], [2., 20., 7.], [3., 30., 7.], [4., 40., 7.]])
ex4_transform = np.array([[1., 0.], [0., 1.], [1., 1.]])
ex4_offset = np.array([[.5, -.5]])
# TODO: mean, scale, standardized, result

## ۵. ابزارهای بقا — اجرای زنده، دقیقه ۶۵ تا ۷۰

سه پرسش ثابت پیش از هر مرحله: **شکل چیست؟ نوع چیست؟ مقدارها معتبرند؟** خطا را در اولین محل نقض قرارداد متوقف کنید.

### ۵.۱ — تشخیص خرابی پیش از اصلاح

NaN یعنی مقدار عددی نامعتبر/ناموجود، Inf یعنی بی‌نهایت. `isfinite` هر دو را پوشش می‌دهد. `clip` فقط بازه را محدود می‌کند؛ NaN را درمان نمی‌کند. جایگزینی با صفر یک تصمیم مسئله است، نه نسخه عمومی برای همه داده‌ها. نخست محل و علت را بررسی کنید.

In [ ]:
damaged = np.array([1., np.nan, np.inf, -np.inf, 10.])
print('NaN:', np.isnan(damaged))
print('Inf:', np.isinf(damaged))
print('finite:', np.isfinite(damaged))
print('clip still contains NaN:', np.clip(damaged, 0, 5))
policy_fixed = np.nan_to_num(damaged, nan=0., posinf=5., neginf=0.)
print('explicit policy:', policy_fixed)
probabilities = np.array([0., .5, 1.])
safe = np.clip(probabilities, 1e-7, 1 - 1e-7)
print('finite logs:', np.log(safe))
assert np.isfinite(np.log(safe)).all()

**مکث مدرس / سؤال از کلاس:** چرا `damaged == np.nan` روش تشخیص NaN نیست؟ محدودسازی احتمال مفروضات محاسبه را تغییر می‌دهد و باید آگاهانه انجام شود.

### ۵.۲ — assert و ذخیره‌سازی

`assert` برای قرارداد و دیباگ آموزشی مناسب است؛ با اجرای بهینه پایتون ممکن است حذف شود. برای اعتبارسنجی ورودی خارجی در برنامه واقعی `if` و `raise` به کار ببرید. `np.save` یک آرایه و `np.savez_compressed` چند آرایه نام‌دار ذخیره می‌کند؛ نام فایل‌ها را معنادار بگذارید.

In [ ]:
def checked_transform(data, matrix):
    assert data.ndim == 2 and matrix.ndim == 2, 'Expected two matrices'
    assert data.shape[1] == matrix.shape[0], f'Wrong shapes: {data.shape}, {matrix.shape}'
    assert np.isfinite(data).all(), 'Non-finite input'
    return data @ matrix

demo_standardized = np.array([[1., -1.], [0., 0.], [-1., 1.]])
print(checked_transform(ex4_data, ex4_transform).shape)
expected_error(AssertionError, lambda: checked_transform(ex4_data, np.ones((4, 2))))
artifact_dir = Path('bootcamp_artifacts')
artifact_dir.mkdir(exist_ok=True)
np.save(artifact_dir / 'standardized.npy', demo_standardized)
restored = np.load(artifact_dir / 'standardized.npy', allow_pickle=False)
np.testing.assert_array_equal(restored, demo_standardized)
print('saved and checked:', restored.shape)